# ✍️ Huấn Luyện Agent 3: Summarizer (`Generator` SFT+DPO & `Critic` Hard-Negatives) Trên Kaggle T4x2
**Chức năng:** Huấn luyện bộ đôi QLoRA (`r=64, max_seq_length=3072`) trên 2 GPU Tesla T4 (`32GB VRAM`):
1. **`generator_qlora`**: Học SFT tổng hợp có trích dẫn `[1]`, `[2]` $\rightarrow$ Căn chỉnh DPO ($\beta=0.1$) chống bịa số liệu.
2. **`critic_qlora`**: Học trên tập Hard Negatives để tự động phát hiện câu thiếu căn cứ và kích hoạt lệnh `"rewrite"` / `"request_more_context"`.


In [ ]:
# BƯỚC 1: CẤU HÌNH GOOGLE DRIVE LINK HOẶC GITHUB TOKEN
GOOGLE_DRIVE_FOLDER_LINK = ""
GITHUB_TOKEN = ""
PART = "both"  # "both" (cả Generator + Critic), hoặc "generator", hoặc "critic"
EPOCHS = 2


In [ ]:
# BƯỚC 2: CLONE GITHUB & CÀI ĐẶT THƯ VIỆN
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate sentencepiece gdown safetensors
import os, subprocess
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"
if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "https://github.com/ducmanh-jr/Multi-Agent_RAG.git", PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)


In [ ]:
# BƯỚC 3: CHẠY HUẤN LUYỆN AGENT 3 (GENERATOR SFT+DPO & CRITIC) TRÊN KAGGLE DUAL T4
!python training_lab/agent3_summarizer/train_agent3_t4x2.py --part {PART} --epochs {EPOCHS}


In [ ]:
# BƯỚC 4: TỰ ĐỘNG LƯU TRỌNG SỐ AGENT 3 VỀ GOOGLE DRIVE HOẶC GITHUB
import shutil
zip_file = shutil.make_archive("agent3_summarizer_weights", "zip", "training_lab/agent3_summarizer/weights")
print("✅ Đã đóng gói:", zip_file)
if GITHUB_TOKEN:
    auth_repo = f"https://{GITHUB_TOKEN}@github.com/ducmanh-jr/Multi-Agent_RAG.git"
    subprocess.run(["git", "config", "user.email", "trainer@kaggle-t4x2.local"], check=False)
    subprocess.run(["git", "config", "user.name", "Kaggle T4x2 Auto Trainer"], check=False)
    subprocess.run(["git", "add", "training_lab/agent3_summarizer/weights"], check=False)
    subprocess.run(["git", "commit", "-m", "feat(agent3): update SFT+DPO generator and critic weights from T4x2"], check=False)
    subprocess.run(["git", "push", auth_repo, "HEAD:main"], check=True)
    print("🎉 Đã tự động đẩy trọng số Agent 3 về GitHub!")
